# Baseline Analysis

In [1]:
import polars as pl

from social_groups.reporting.analysis_columns import AnalysisColumn
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
)
from social_groups.reporting.parsing import (
    AnswerComparer,
    AnswerOptions,
    AnswerParser,
)

/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagstermill/manager.py:398: BetaWarning: Class `Manager` is currently in beta, and may have breaking changes in minor version releases, with behavior changes in patch releases.
  MANAGER_FOR_NOTEBOOK_INSTANCE = Manager()
/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/src/social_groups/analysis/notebook_assets.py:136: BetaWarning: Class `LocalFileCodeReference` is currently in beta, and may have breaking changes in minor version releases, with behavior changes in patch releases.
  dg.LocalFileCodeReference(
/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/src/social_groups/analysis/notebook_assets.py:141: BetaWarning: Class `LocalFileCodeReference` is currently in beta, and may have breaking changes in minor version releases, with behavior changes in patch releases.
  dg.LocalFileCodeReference(
/Users/philipp/Documents/Studium/Informatik/Masterth

In [2]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling="wrong"
)

In [3]:
from social_groups.analysis.definitions import defs

baseline_frame = defs().load_asset_value("baseline")

/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagster/_config/pythonic_config/typing_utils.py:111: UserWarning: Field name "extension" in "PolarsParquetIOManager" shadows an attribute in parent "BasePolarsUPathIOManager"
  return super().__new__(cls, name, bases, namespaces, **kwargs)
2026-03-10 13:23:11 +0800 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/baseline.parquet using PolarsParquetIOManager...


In [4]:
baseline_frame.head()

id,run_id,question_id,phoenix_span_url,run_identifier,final_answer,original_question_id,category,question,answer_string,model_name
i64,i64,i64,str,str,str,i64,str,str,str,str
3920,40,60,"""http://localhost:6006/projects…","""2026-03-08-03-07-45 - heteroge…","""The passage states that perest…",4888,"""history""","""Q: This question refers to the…","""H""","""Qwen/Qwen3-14B"""
3921,40,65,"""http://localhost:6006/projects…","""2026-03-08-03-07-45 - heteroge…","""The answer is (D): Growth in b…",3249,"""biology""","""Q: How does the term ""growth"" …","""D""","""Qwen/Qwen3-14B"""
3922,40,63,"""http://localhost:6006/projects…","""2026-03-08-03-07-45 - heteroge…","""The answer is **(C): Aortic di…",6037,"""health""","""Q: A 47-year-old man is brough…","""C""","""Qwen/Qwen3-14B"""
3923,40,66,"""http://localhost:6006/projects…","""2026-03-08-03-07-45 - heteroge…","""The answer is (A): unethically…",2626,"""psychology""","""Q: A non-custodial parent asks…","""J""","""Qwen/Qwen3-14B"""
3924,40,62,"""http://localhost:6006/projects…","""2026-03-08-03-07-45 - heteroge…","""The Islamic understanding of j…",11057,"""philosophy""","""Q: Which of the following is N…","""E""","""Qwen/Qwen3-14B"""


### Number of unparsable answers

In [5]:
(
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .group_by("model_name")
    .agg(
        no_null=pl.col(AnalysisColumn.parsed_answer.value)
        .str.starts_with("___")
        .not_()
        .sum(),
        null_percentage=(
            pl.col(AnalysisColumn.parsed_answer.value).str.starts_with("___").mean()
            * 100
        ).round(2),
    )
)

model_name,no_null,null_percentage
str,u32,f64
"""Qwen/Qwen3-14B""",380,5.0
"""Qwen/Qwen3-0.6B""",338,15.5
"""Qwen/Qwen3-4B""",379,5.25


### Accuracy per Model

In [6]:
from social_groups.analysis.defs.notebooks.definitions import register_materialization
accuracy_per_model = (
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .with_columns(
        is_correct=comparer(
            pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
        )
    )
    .group_by("model_name")
    .agg(accuracy=pl.col("is_correct").mean())
    .sort(pl.col("model_name").str.extract(r"-(\d+\.?\d*)B", 1).cast(pl.Float64))
)

register_materialization("baseline_full_evaluation_table", accuracy_per_model, description="Accuracy Per Model evaluated on the full dataset.")


accuracy_per_model

Skipping Materialization because in interactive mode.


/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagstermill/manager.py:286: BetaWarning: Class `DagstermillExecutionContext` is currently in beta, and may have breaking changes in minor version releases, with behavior changes in patch releases.
  self.context = DagstermillExecutionContext(
/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagster/_core/execution/context_creation_job.py:276: RuntimeWarning: coroutine 'BaseEventLoop.shutdown_asyncgens' was never awaited
  pass


model_name,accuracy
str,f64
"""Qwen/Qwen3-0.6B""",0.3025
"""Qwen/Qwen3-4B""",0.4925
"""Qwen/Qwen3-14B""",0.62
